In [30]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import pickle

In [7]:
ufos = pd.read_csv('./data/ufos.csv')
ufos.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [8]:
ufos.columns


Index(['datetime', 'city', 'state', 'country', 'shape', 'duration (seconds)',
       'duration (hours/min)', 'comments', 'date posted', 'latitude',
       'longitude'],
      dtype='object')

In [9]:
ufos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              80332 non-null  object 
 1   city                  80332 non-null  object 
 2   state                 74535 non-null  object 
 3   country               70662 non-null  object 
 4   shape                 78400 non-null  object 
 5   duration (seconds)    80332 non-null  float64
 6   duration (hours/min)  80332 non-null  object 
 7   comments              80317 non-null  object 
 8   date posted           80332 non-null  object 
 9   latitude              80332 non-null  float64
 10  longitude             80332 non-null  float64
dtypes: float64(3), object(8)
memory usage: 6.7+ MB


In [10]:
ufos_clean = pd.DataFrame({
    'Seconds': ufos['duration (seconds)'],
    'Country': ufos['country'],
    'Latitude': ufos['latitude'],
    'Longitude': ufos['longitude']
})

ufos_clean.head()

,Seconds,Country,Latitude,Longitude
0,2700.0,us,29.883056,-97.941111
1,7200.0,NaN,29.384210,-98.581082
2,20.0,gb,53.200000,-2.916667
3,20.0,us,28.978333,-96.645833
4,900.0,us,21.418056,-157.803611


In [11]:
ufos_clean.isnull().sum()

Seconds         0
Country      9670
Latitude        0
Longitude       0
dtype: int64

In [14]:
ufos_clean.dropna(subset=['Country'],inplace=True)
ufos_clean.isnull().sum()

Seconds      0
Country      0
Latitude     0
Longitude    0
dtype: int64

In [15]:
ufos_clean = ufos_clean[(ufos_clean['Seconds'] >= 1) & (ufos_clean['Seconds'] <= 60)]

ufos_clean.describe()

,Seconds,Latitude,Longitude
count,25863.000000,25863.000000,25863.000000
mean,23.704296,38.668119,-90.862549
std,21.307403,8.852130,31.882091
min,1.000000,-42.883209,-170.478889
25%,5.000000,34.257500,-114.036667
50%,15.000000,39.515000,-89.093889
75%,40.000000,42.865000,-79.792222
max,60.000000,72.700000,153.593918


In [17]:
encoder = LabelEncoder()
ufos_clean['Country'] = encoder.fit_transform(ufos_clean['Country'])

ufos_clean.head()

,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
23,60.0,4,45.582778,-122.352222
24,3.0,3,51.783333,-0.783333


In [23]:
X = ufos_clean.drop('Country', axis=1)
y = ufos_clean['Country']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state=42)


In [26]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [29]:
predictions = model.predict(X_test)

print(classification_report(y_test, predictions))
print("Accuracy:", accuracy_score(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.83      0.43      0.56       288
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00       134
           4       0.97      0.99      0.98      4700

    accuracy                           0.96      5173
   macro avg       0.96      0.88      0.91      5173
weighted avg       0.96      0.96      0.96      5173

Accuracy: 0.9630775178813068


In [31]:
model_filename = "ufo-model.pkl"

with open(model_filename, "wb") as file:
    pickle.dump(model, file)

In [36]:
with open("ufo-model.pkl", "rb") as file:
    loaded_model = pickle.load(file)
test_prediction = loaded_model.predict([[50, 44, -12]])

print('the country is :',int(test_prediction))

the country is : 3


c:\Users\mario\Documents\DSMLAISL LEARNING PATH\PY REPOSITORIES\Labs\lab-model-deployment\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\mario\AppData\Local\Temp\ipykernel_27376\2521323109.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('the country is :',int(test_prediction))
